# ROOTS Step 3: Compound Library Curation (Python)

**Input:** Validated structures from Step 2 (for context only; this step curates the compound library independently).

**What this notebook does:**
1. Downloads the COCONUT natural-product database
2. Applies Ro5, Veber, and PAINS filtering via RDKit
3. Caps the filtered library at ~500-2,000 compounds
4. Runs a stereochemistry-integrity check on 3D embedding
5. Saves the final library to Drive for Step 4 (receptor/ligand prep)

## 0. Environment setup and Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

DRIVE_BASE = Path("/content/drive/MyDrive/ROOTS_Pipeline")
STEP3_DIR = DRIVE_BASE / "Step3"
STEP3_DIR.mkdir(parents=True, exist_ok=True)
(STEP3_DIR / "logs").mkdir(exist_ok=True)

print(f"Timestamp: {datetime.now().isoformat()}")

In [ ]:
%%capture
!pip install -q rdkit-pypi requests

## 1. COCONUT database retrieval

The raw download is archived at retrieval time. COCONUT updates without versioning; an unarchived pull is not reproducible.

In [ ]:
import requests

# COCONUT provides bulk download via its API
COCONUT_API = "https://coconut.naturalproducts.net/api"

print("Querying COCONUT API for available downloads...")
# Try the COCONUT REST API for SMILES export
# Note: COCONUT's API and download format may change; verify at runtime

try:
    # Attempt to get compound count first
    resp = requests.get(f"{COCONUT_API}/search/simple?query=*&limit=1", timeout=30)
    if resp.status_code == 200:
        data = resp.json()
        total = data.get('total', data.get('count', 'unknown'))
        print(f"COCONUT reports {total} total compounds")
    else:
        print(f"COCONUT API response: {resp.status_code}")
except Exception as e:
    print(f"COCONUT API query: {e}")
    print("Proceeding with bulk download approach...")

# Download COCONUT SMILES dataset
# COCONUT provides a downloadable SDF/CSV file
COCONUT_DOWNLOAD_URL = "https://coconut.naturalproducts.net/download/coconut_smiles.csv"

print(f"\nDownloading COCONUT SMILES from: {COCONUT_DOWNLOAD_URL}")
print("(This may take several minutes for the full database)")

try:
    coconut_resp = requests.get(COCONUT_DOWNLOAD_URL, timeout=600, stream=True)
    if coconut_resp.status_code == 200:
        raw_path = STEP3_DIR / "coconut_raw.csv"
        with open(raw_path, 'wb') as f:
            for chunk in coconut_resp.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Raw COCONUT data saved to: {raw_path}")

        # Load and inspect
        coconut_df = pd.read_csv(raw_path)
        print(f"COCONUT loaded: {len(coconut_df)} compounds, {len(coconut_df.columns)} columns")
        print(f"Columns: {list(coconut_df.columns)}")
    else:
        print(f"Download failed: {coconut_resp.status_code}")
        print("Attempting alternative COCONUT download endpoints...")
        # Try alternative URL patterns
        alt_urls = [
            "https://coconut.naturalproducts.net/download/smiles",
            "https://coconut.naturalproducts.net/api/download/coconut"
        ]
        coconut_df = None
        for url in alt_urls:
            try:
                r = requests.get(url, timeout=120)
                if r.status_code == 200:
                    raw_path = STEP3_DIR / "coconut_raw.csv"
                    with open(raw_path, 'wb') as f:
                        f.write(r.content)
                    coconut_df = pd.read_csv(raw_path)
                    print(f"Downloaded from {url}: {len(coconut_df)} compounds")
                    break
            except:
                continue
        if coconut_df is None:
            print("\nCOCONUT download failed from all endpoints.")
            print("Manual action required: download COCONUT from https://coconut.naturalproducts.net/download")
            print(f"Save the SMILES CSV to: {STEP3_DIR / 'coconut_raw.csv'}")
            print("Then re-run from the next cell.")
except Exception as e:
    print(f"Download error: {e}")
    print("Manual download may be required.")

# Record retrieval timestamp
retrieval_log = {
    'retrieval_timestamp': datetime.now().isoformat(),
    'source_url': COCONUT_DOWNLOAD_URL,
    'note': 'COCONUT updates without versioning; this raw file is the archived pull'
}
with open(STEP3_DIR / "logs" / "coconut_retrieval.json", 'w') as f:
    json.dump(retrieval_log, f, indent=2)

## 2. Drug-likeness filtering: Ro5, Veber, PAINS

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, FilterCatalog, AllChem
from rdkit.Chem.FilterCatalog import FilterCatalogParams

# Load COCONUT data
raw_path = STEP3_DIR / "coconut_raw.csv"
if not raw_path.exists():
    raise FileNotFoundError(f"COCONUT raw data not found at {raw_path}. Download it first.")

coconut_df = pd.read_csv(raw_path)

# Identify the SMILES column (may be named differently across COCONUT versions)
smiles_col = None
for candidate in ['canonical_smiles', 'SMILES', 'smiles', 'Canonical_SMILES', 'CANONICAL_SMILES']:
    if candidate in coconut_df.columns:
        smiles_col = candidate
        break

if smiles_col is None:
    print(f"Available columns: {list(coconut_df.columns)}")
    raise ValueError("Cannot identify SMILES column in COCONUT data")

print(f"Using SMILES column: {smiles_col}")
print(f"Starting compounds: {len(coconut_df)}")

# Parse SMILES to RDKit molecules
coconut_df['mol'] = coconut_df[smiles_col].apply(lambda s: Chem.MolFromSmiles(str(s)) if pd.notna(s) else None)
valid = coconut_df['mol'].notna()
print(f"Valid SMILES parsed: {valid.sum()} / {len(coconut_df)}")
coconut_df = coconut_df[valid].copy()

In [ ]:
# Compute descriptors for Ro5 and Veber filters
coconut_df['MW'] = coconut_df['mol'].apply(Descriptors.MolWt)
coconut_df['LogP'] = coconut_df['mol'].apply(Descriptors.MolLogP)
coconut_df['HBD'] = coconut_df['mol'].apply(Descriptors.NumHDonors)
coconut_df['HBA'] = coconut_df['mol'].apply(Descriptors.NumHAcceptors)
coconut_df['TPSA'] = coconut_df['mol'].apply(Descriptors.TPSA)
coconut_df['RotBonds'] = coconut_df['mol'].apply(Descriptors.NumRotatableBonds)

# Lipinski Rule of Five
ro5_mask = (
    (coconut_df['MW'] <= 500) &
    (coconut_df['LogP'] <= 5) &
    (coconut_df['HBD'] <= 5) &
    (coconut_df['HBA'] <= 10)
)
print(f"Pass Ro5: {ro5_mask.sum()} / {len(coconut_df)}")

# Veber rules
veber_mask = (
    (coconut_df['TPSA'] <= 140) &
    (coconut_df['RotBonds'] <= 10)
)
print(f"Pass Veber: {veber_mask.sum()} / {len(coconut_df)}")

# Combined Ro5 + Veber
ro5_veber = ro5_mask & veber_mask
print(f"Pass Ro5 + Veber: {ro5_veber.sum()} / {len(coconut_df)}")

In [ ]:
# PAINS filter
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog.FilterCatalog(params)

def has_pains(mol):
    """Returns True if molecule contains PAINS substructure (should be excluded)."""
    if mol is None:
        return True
    return pains_catalog.HasMatch(mol)

coconut_df['has_pains'] = coconut_df['mol'].apply(has_pains)
pains_clean = ~coconut_df['has_pains']
print(f"Pass PAINS filter (no PAINS alerts): {pains_clean.sum()} / {len(coconut_df)}")

# Combined: Ro5 + Veber + PAINS-free
all_filters = ro5_veber & pains_clean
filtered_df = coconut_df[all_filters].copy()
print(f"\nAll filters combined: {len(filtered_df)} compounds survive")

## 3. Library size cap

Filtering is applied before capping (not after), to avoid an artificially shrunken final set. If the filtered pool exceeds the cap, a seeded random subsample is taken.

In [ ]:
LIBRARY_CAP = 2000
SEED = 42

if len(filtered_df) > LIBRARY_CAP:
    print(f"Filtered pool ({len(filtered_df)}) exceeds cap ({LIBRARY_CAP}). Subsampling with seed={SEED}.")
    filtered_df = filtered_df.sample(n=LIBRARY_CAP, random_state=SEED)
elif len(filtered_df) == 0:
    raise ValueError("No compounds survived filtering. Check COCONUT data and filter thresholds.")
else:
    print(f"Filtered pool ({len(filtered_df)}) is within cap ({LIBRARY_CAP}). Using full set.")

print(f"Final library size: {len(filtered_df)}")

## 4. Stereochemistry-integrity check on 3D embedding

Defined stereocentres in the SMILES must survive 3D embedding. Compounds where stereocentres are lost during embedding are excluded and logged.

In [ ]:
from rdkit.Chem import AllChem, FindMolChiralCenters

EMBED_SEED = 42

def check_stereochemistry(mol, seed=EMBED_SEED):
    """
    Embed molecule in 3D and verify stereocentres survive.
    Returns (success, original_centers, embedded_centers, match).
    """
    if mol is None:
        return False, [], [], False

    # Record original stereocentres
    original_centers = FindMolChiralCenters(mol, includeUnassigned=True)

    # Embed in 3D
    mol_3d = Chem.AddHs(mol)
    result = AllChem.EmbedMolecule(mol_3d, AllChem.ETKDGv3())
    if result == -1:
        # Embedding failed entirely
        return False, original_centers, [], False

    # Optimise with MMFF94
    try:
        AllChem.MMFFOptimizeMolecule(mol_3d, maxIters=500)
    except:
        pass

    # Re-derive stereocentres from 3D
    Chem.AssignStereochemistryFrom3D(mol_3d)
    embedded_centers = FindMolChiralCenters(mol_3d, includeUnassigned=True)

    # Compare: all original defined centres must be preserved
    if len(original_centers) == 0:
        return True, original_centers, embedded_centers, True

    orig_dict = {idx: chirality for idx, chirality in original_centers if chirality != '?'}
    embed_dict = {idx: chirality for idx, chirality in embedded_centers if chirality != '?'}

    match = all(
        embed_dict.get(idx) == chirality
        for idx, chirality in orig_dict.items()
    )

    return True, original_centers, embedded_centers, match

print("Running stereochemistry-integrity check (this may take a while)...")
stereo_results = []
for idx, row in filtered_df.iterrows():
    success, orig, embed, match = check_stereochemistry(row['mol'])
    stereo_results.append({
        'idx': idx,
        'embed_success': success,
        'original_stereocenters': len(orig),
        'stereo_preserved': match
    })

stereo_df = pd.DataFrame(stereo_results)
embed_ok = stereo_df['embed_success'] & stereo_df['stereo_preserved']

print(f"\nStereochemistry check results:")
print(f"  Embedding successful: {stereo_df['embed_success'].sum()} / {len(stereo_df)}")
print(f"  Stereocentres preserved: {stereo_df['stereo_preserved'].sum()} / {len(stereo_df)}")
print(f"  Compounds passing both: {embed_ok.sum()} / {len(stereo_df)}")

# Exclude failures
filtered_df = filtered_df.loc[stereo_df[embed_ok]['idx']].copy()
print(f"\nFinal library after stereochemistry check: {len(filtered_df)}")

## 5. Save final library

In [ ]:
# Export final library
output_cols = [smiles_col, 'MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds']
# Add any ID column if present
id_cols = [c for c in filtered_df.columns if 'id' in c.lower() or 'name' in c.lower() or 'coconut' in c.lower()]
output_cols = id_cols + output_cols

# Keep only existing columns
output_cols = [c for c in output_cols if c in filtered_df.columns]
library_out = filtered_df[output_cols].copy()

output_path = STEP3_DIR / "library_final.csv"
library_out.to_csv(output_path, index=False)
print(f"Final library saved to: {output_path}")
print(f"  Compounds: {len(library_out)}")
print(f"  Columns: {list(library_out.columns)}")

# Save attrition report
attrition = {
    'timestamp': datetime.now().isoformat(),
    'coconut_raw_count': int(len(coconut_df)) if 'coconut_df' in dir() else 'unknown',
    'after_smiles_parse': int(valid.sum()) if 'valid' in dir() else 'unknown',
    'after_ro5': int(ro5_mask.sum()),
    'after_ro5_veber': int(ro5_veber.sum()),
    'after_pains': int(all_filters.sum()),
    'after_cap': LIBRARY_CAP,
    'after_stereochem': int(len(filtered_df)),
    'final_count': int(len(library_out)),
    'seed': SEED,
    'filters_applied': ['Lipinski Ro5', 'Veber', 'PAINS', f'Cap at {LIBRARY_CAP}', 'Stereochemistry integrity'],
    'scope_note': 'Ro5/Veber filtering applied to NP library is a stated scope limitation; NPs legitimately violate these rules'
}

with open(STEP3_DIR / "logs" / "step3_attrition.json", 'w') as f:
    json.dump(attrition, f, indent=2)

print(f"\nAttrition report saved.")
print(f"\n=== Step 3 complete ===")
print(f"Step 4 (receptor/ligand prep) will load: {output_path}")
print(f"and structures from: {DRIVE_BASE / 'Step2' / 'structures'}")